In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from scipy.stats import spearmanr

Also, Polymarket has a practical advantage over Binance: you are not competing against the same level of ultra-low-latency 
firms trading microseconds. The bottleneck is usually information and risk management, not shaving nanoseconds.

Given your existing C++ engine, I would actually approach it incrementally:

Port your order book + execution engine.
Implement YES-only quoting.
Add inventory skew.
Add microprice/trade imbalance.
Add a simple options-implied probability anchor.
Only then build ML.

The hardest part is not writing the code — it is deciding what your fair value should be. 
Your existing market-making infrastructure is already the part many people struggle to build.

In [ ]:
This is the hardest part of building a Polymarket probability model. You are correct that contracts constantly change:

"BTC above $150k by Dec 31"
"Fed cuts rates before March"
"Team X wins championship"
"Candidate Y wins election"

Each has different dynamics.

The key is: you usually do not train one model per contract. You train a general probability model, then calibrate its outputs.

1. The model should predict probabilities, not prices

Your XGBoost target is:

Did the event happen?

Example historical dataset:

timestamp	BTC price	vol	days left	funding	target
Jan 1	95k	50%	90	0.01	1
Jan 2	97k	55%	89	0.02	1
Jan 3	92k	60%	88	-0.01	0

The model learns:

P(BTC > strike at expiry)

It does not care about the contract ID

In [ ]:
2. Use features that describe the contract

You need the model to understand what kind of bet it is.

For a BTC price event:

struct EventFeatures {
    double spot_price;
    double strike_price;
    double distance_to_strike;
    double implied_vol;
    double realized_vol;
    double time_to_expiry;
    double funding;
};

A very important feature:

distance_to_strike =
    (spot - strike) / spot;

Example:

Contract A:

BTC > 150k
BTC = 100k

distance = -50%

Contract B:

BTC > 110k
BTC = 100k

distance = -10%

They are completely different situations.

In [ ]:
3. Calibration is done on historical predictions

Suppose your model produces:

prediction = 0.70

You collect many historical examples.

Example:

Model predicted 0.70:

Contract 1 -> won
Contract 2 -> won
Contract 3 -> lost
Contract 4 -> won
...

After 1000 samples:

700 actually won

Great:

predicted: 70%
actual:    70%

The model is calibrated.

But maybe you see:

Predicted 70%

Actual:
58%

Your model is overconfident.

You calibrate it:

raw model output:

0.70

calibrator:

0.62

Now your trading price is:

YES fair = 0.62

In [ ]:
4. Use a separate calibration layer

Do not retrain the whole model.

Your pipeline becomes:

features
   |
   v
XGBoost
   |
   v
raw probability
   |
   v
calibration function
   |
   v
final probability

Example:

double raw = model.predict(features);

double calibrated =
    calibrator.transform(raw);

Common calibrators:

Isotonic regression

Good when you have lots of data.

Learns:

raw probability
       |
       v
actual probability

Example:

0.50 -> 0.48
0.60 -> 0.57
0.70 -> 0.65
0.80 -> 0.74
Platt scaling

Fits:

calibrated =
sigmoid(a * raw + b)

Good when data is limited.

In [ ]:
5. But contracts change — how do you avoid overfitting?

You should train across many contracts.

For example:

Bad:

Train:
BTC > 150k Dec 2026

You have only one sample.

Better:

Train:
BTC > 80k
BTC > 100k
BTC > 120k
BTC > 150k

Across:
2020-2026

The model learns the general relationship:

spot
strike
volatility
time
=
probability

In [ ]:
6. For crypto events, you can start with an options model

Actually, for BTC price threshold markets, you have a huge advantage.

A contract:

BTC above $150k at expiry

is basically a binary option.

The theoretical probability is approximately:

P(BTC_T > strike)

which can come from:

Black-Scholes style models
volatility surface
options implied volatility

You can use this as your baseline.

Example:

Options market implies:

BTC > 150k probability = 32%

Your XGBoost model learns residuals:

Polymarket probability
-
options probability

Similar to your current residual model.

In [ ]:
7. How I would architect it with your existing code

You already have:

ResidualModel
    Features
    XGBoost
    prediction

I would create:

ProbabilityModel

Features:
    spot
    strike
    expiry
    volatility
    funding
    options_probability

Output:
    calibrated_probability

Then:

YES fair:

probability_model
        +
microstructure_residual
        -
inventory_skew

Something like:

double fair =
    probability_model.predict(event)
    +
    mm_residual_model.predict(book_features)
    -
    inventory_skew;

In [ ]:
For that dataset, you need to reconstruct it yourself. There is no single API that gives you:

timestamp | BTC price | volatility | days left | funding | target

because that table is not raw market data — it is a machine-learning training dataset you create from raw data.

The process is:

Raw BTC market data
        |
        v
Feature engineering
        |
        v
Label generation
        |
        v
Training dataset
1. BTC price history

You need historical BTC candles/trades.

Sources:

Binance historical klines
Coinbase
Bybit
OKX

You need:

timestamp
open
high
low
close
volume

Example:

2025-01-01 00:00
BTC close = 95,000

2025-01-02 00:00
BTC close = 97,000
2. Volatility

You calculate this.

Example:

Daily returns:

r = log(price_today / price_yesterday)

Then:

30-day volatility =
stddev(last 30 returns) * sqrt(365)

Your dataset becomes:

timestamp   BTC     vol
Jan 1       95000   50%
Jan 2       97000   55%
3. Days left

This comes from the contract.

Example Polymarket contract:

BTC above $150,000 by Dec 31, 2026

At every historical timestamp:

expiry = Dec 31 2026

days_left =
expiry - timestamp

So:

Jan 1 2026
days_left = 364

Jun 1 2026
days_left = 213
4. Funding rate

This comes from perpetual futures exchanges.

Example:

Binance Futures gives:

BTCUSDT perpetual funding

timestamp
funding_rate

Example:

Jan 1
funding = 0.01%

Jan 2
funding = 0.02%
5. The target (most important)

This is where you create the label.

For:

BTC above $150k on expiry?

At every historical point you ask:

"What happened at expiry?"

Example:

Contract:

BTC > 150000
expiry = Dec 31 2025

At:

Jan 1 2025

You record:

BTC = 95000
vol = 50%
days_left = 364

Then look forward:

Dec 31 2025 BTC = 160000

Therefore:

target = 1

Another example:

At:

Mar 1 2025
BTC = 90000

At expiry:

BTC = 140000

Target:

0

Your final dataset:

timestamp    spot    strike   vol   days_left   funding   target

Jan 1        95000   150000   50%   364         0.01      1
Feb 1        98000   150000   55%   333         0.02      1
Mar 1        90000   150000   60%   305        -0.01      0

In [ ]:
Professional-style architecture

Something like:

                Deribit
                  |
                  v
          implied distribution
                  |
                  v
          option fair probability
                  |
                  |
Polymarket book ---> microstructure model
                  |
                  v
             final fair value
                  |
                  v
              quotes

In [ ]:
Where do you get the Polymarket contracts?

You need historical markets:

contract question
strike (if it is a price event)
expiry date
resolution outcome

Polymarket has APIs/data endpoints for market metadata and historical markets.

You would collect:

market_id
question
end_date
resolution

Example:

market:
"Will BTC be above $150k on Dec 31?"

end_date:
2025-12-31

resolved:
YES

Then join it with BTC historical prices.

For crypto event markets specifically

You actually have a shortcut.

A BTC price binary event is basically a digital option.

You can generate synthetic training examples.

Example:

Pick random historical dates:

BTC price = 80k
strike = 100k
expiry = 60 days

Ask:

Did BTC exceed 100k within 60 days?

Label:

yes/no

Generate millions of samples.

Features:

struct EventFeatures {
    double spot;
    double strike;
    double distance;
    double volatility;
    double days_left;
    double funding;
};

This gives you a model that generalizes to new Polymarket contracts.

For your architecture, I would probably do:

Historical BTC candles
        +
BTC options implied volatility
        +
Funding rates
        |
        v
XGBoost classifier
        |
        v
P(BTC > strike at expiry)
        |
        v
YES fair value

Then your existing market-making model sits on top:

YES probability model
          +
order book microstructure model
          +
inventory skew
          |
          v
quotes

The nice part is you don't need to wait for Polymarket history. You can build a strong first version entirely from BTC historical data and synthetic BTC threshold contracts.

In [ ]:
What I mean is: you can create your own fake Polymarket contracts from historical BTC data and use them as training examples.

You do not need to wait for Polymarket to have thousands of historical BTC prediction markets.

A Polymarket contract like:

"Will BTC be above $100,000 on June 30?"

is just a binary question:

YES = BTC_final > 100000
NO  = BTC_final <= 100000

So you can generate millions of similar questions from BTC history.

Example

Suppose you have BTC daily prices:

Date          BTC close

2022-01-01    47000
2022-01-02    47300
...
2023-01-01    16600
...
2024-03-01    62000
...

Now randomly pick a date.

Sample 1

Pick:

start date = 2022-01-01

BTC price:

47000

Now randomly create a hypothetical contract:

Strike = 60000
Expiry = 90 days later

Your fake Polymarket question:

"Will BTC be above $60k in 90 days?"

Your features:

spot              = 47000
strike            = 60000
distance          = -27.6%
volatility        = 70%
days_left         = 90

Now look forward in your historical data:

90 days later:

2022-04-01 BTC = 46000

Did BTC exceed 60000?

No.

Label:

target = 0

Your training row:

spot	strike	vol	days	target
47000	60000	70%	90	0
Sample 2

Pick another random date:

start = 2020-10-01
BTC = 10500

Generate:

strike = 15000
expiry = 180 days

Question:

"Will BTC be above $15k in 180 days?"

Look forward:

2021-04-01 BTC = 59000

Yes.

Label:

target = 1

Dataset:

spot	strike	vol	days	target
10500	15000	80%	180	1
After doing this millions of times

Your dataset becomes:

spot     strike    vol    days_left    funding    target

47000    60000     70%       90          0.01        0
10500    15000     80%       180         0.02        1
62000    90000     50%       365        -0.01        1
30000    25000     90%       30          0.03        0
...

Then train:

XGBClassifier(
    features =
    [
      spot,
      strike,
      volatility,
      days_left,
      funding
    ],

    target =
    [
      0 or 1
    ]
)

The model learns:

Given the current BTC state, what is the probability BTC finishes above the strike?

Why this works

Because a Polymarket crypto contract is not really unique.

These two contracts:

BTC > 100k in 60 days
BTC > 150k in 180 days

are different questions, but mathematically they are the same structure:

current price
+
distance to strike
+
time
+
volatility
=
probability distribution of future price